In [ ]:
from src.data.data import *
from src.embedor import *
from src.plotting import *
import pandas as pd
import matplotlib
import seaborn as sns
import argparse
import umap
import numpy as np
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding
import phate
import json
from src.utils import *
import os

from scipy.stats import t, ttest_rel




In [ ]:
# load full experiment

import pickle as pkl

datasets = ['mnist', 'fmnist', 'developmental', 'macosko', 'chimp']

plt.rcParams.update({'font.size': 12})

for dataset in datasets:
    print("DATASET: ", dataset)
    with open(f'../../outputs/server_experiments/umap_tsne_param_ablations/umap_tsne_param_ablation_{dataset}.pkl', 'rb') as f: 
        # output from scripts/umap_tsne_param_ablation.py
        results = pkl.load(f)

    output_dir = 'figures'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    ablation_dir = os.path.join(output_dir, 'ablation')
    if not os.path.exists(ablation_dir):
        os.makedirs(ablation_dir)
    dataset_dir = os.path.join(ablation_dir, dataset)
    if not os.path.exists(dataset_dir):
        os.makedirs(dataset_dir)

    if dataset =='developmental':
        yticks = np.arange(0.3, 0.75, 0.05)  # 0.3 to 0.7 inclusive
        plt.rcParams.update({'font.size': 12})
        # Extract embedor correlations
        embedor_spearman = results['embedor']['spearman_corr']
        embedor_pearson = results['embedor']['pearson_corr']
        print(embedor_spearman, embedor_pearson)

        # Extract UMAP min_dist correlations
        umap_min_dist_experiment = results['min_dist']
        min_dists = []
        spearman = []
        pearson = []

        for min_dist_str, data in umap_min_dist_experiment.items():
            min_dist = float(min_dist_str.replace('min_dist_', ''))
            min_dists.append(min_dist)
            spearman.append(data['spearman_corr'])
            pearson.append(data['pearson_corr'])

        # Sort by min_dist
        sorted_indices = sorted(range(len(min_dists)), key=lambda i: min_dists[i])
        min_dists = [min_dists[i] for i in sorted_indices]
        spearman = [spearman[i] for i in sorted_indices]
        pearson = [pearson[i] for i in sorted_indices]
        print(max(spearman), max(pearson))

        # Create plot
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # Plot Spearman
        color1 = 'tab:blue'
        ax1.set_xlabel('UMAP min_dist', fontsize=16)
        ax1.set_ylabel('Spearman Correlation', color=color1, fontsize=16)
        ax1.tick_params(axis='y', labelcolor=color1)

        # Twin axis for Pearson
        ax2 = ax1.twinx()
        color2 = 'tab:green'
        ax2.set_ylabel('Pearson Correlation', color=color2, fontsize=16)
        line3 = ax2.plot(min_dists, pearson, marker='s', color=color2, label='UMAP Pearson', linewidth=3, markersize=8)
        line4 = ax2.axhline(y=embedor_pearson, color=color2, linestyle='--', label='EmbedOR Pearson', linewidth=3, markersize=8)
        ax2.tick_params(axis='y', labelcolor=color2)

        line1 = ax1.plot(min_dists, spearman, marker='o', color=color1, label='UMAP Spearman', linewidth=3, markersize=8)
        line2 = ax1.axhline(y=embedor_spearman, color=color1, linestyle='--', label='EmbedOR Spearman', linewidth=3, markersize=8)

        # Align y-axis ticks: 0.3 to 0.7 in steps of 0.05
        yticks = np.arange(0.25, 0.75, 0.05)
        xticks = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
        ax1.set_yticks(yticks)
        ax1.set_yticklabels([f'{tick:.2f}' for tick in yticks])
        ax1.set_ylim(0.25, 0.7)
        ax1.set_xticks(xticks)
        ax2.set_ylim(0.25, 0.7)
        ax2.set_yticks(yticks)
        ax2.set_yticklabels([f'{tick:.2f}' for tick in yticks])
        ax2.set_xticks(xticks)
        ax2.grid(False)

        # Combine legends
        lines = line1 + [line2] + line3 + [line4]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right', framealpha=1.0, facecolor='white', edgecolor='black')
        ax2.legend(lines, labels, loc='upper right', framealpha=1.0, facecolor='white', edgecolor='black')
        plt.tight_layout()
        plt.savefig(os.path.join(dataset_dir, 'umap_min_dist_developmental.png'), dpi=1200, bbox_inches='tight')
        # plt.show()

        # repeat for nsr instead of min_dist
        nsr_experiment = results['nsr']
        nsrs = []
        spearman = []
        pearson = []
        for nsr_str, data in nsr_experiment.items():
            nsr = float(nsr_str.replace('nsr_', ''))
            nsrs.append(nsr)
            spearman.append(data['spearman_corr'])
            pearson.append(data['pearson_corr'])
        # Sort by nsr
        sorted_indices = sorted(range(len(nsrs)), key=lambda i: nsrs[i])
        nsrs = [nsrs[i] for i in sorted_indices]
        spearman = [spearman[i] for i in sorted_indices]
        pearson = [pearson[i] for i in sorted_indices]
        print(max(spearman), max(pearson))

        # Create plot
        fig, ax1 = plt.subplots(figsize=(10, 6))
        # Plot Spearman
        color1 = 'tab:blue'
        ax1.set_xlabel('UMAP negative sampling rate', fontsize=16)
        ax1.set_ylabel('Spearman Correlation', color=color1, fontsize=16)
        ax1.tick_params(axis='y', labelcolor=color1)
        # Twin axis for Pearson
        ax2 = ax1.twinx()
        color2 = 'tab:green'
        ax2.set_ylabel('Pearson Correlation', color=color2, fontsize=16)
        line3 = ax2.plot(nsrs, pearson, marker='s', color=color2, label='UMAP Pearson', linewidth=3, markersize=8)
        line4 = ax2.axhline(y=embedor_pearson, color=color2, linestyle='--', label='EmbedOR Pearson', linewidth=3, markersize=8)
        ax2.tick_params(axis='y', labelcolor=color2)
        line1 = ax1.plot(nsrs, spearman, marker='o', color=color1, label='UMAP Spearman', linewidth=3, markersize=8)
        line2 = ax1.axhline(y=embedor_spearman, color=color1, linestyle='--', label='EmbedOR Spearman', linewidth=3, markersize=8)
        # Align y-axis ticks: 0.3 to 0.7 in steps of 0.05
        yticks = np.arange(0.25, 0.75, 0.05)
        xticks = nsrs
        ax1.set_yticks(yticks)
        ax1.set_ylim(0.25, 0.7)
        ax1.set_xticks(xticks)
        ax2.set_ylim(0.25, 0.7)
        ax2.grid(False)
        # Combine legends
        lines = line1 + [line2] + line3 + [line4]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right', framealpha=1.0, facecolor='white', edgecolor='black')
        ax2.legend(lines, labels, loc='upper right', framealpha=1.0, facecolor='white', edgecolor='black')
        plt.tight_layout()
        plt.savefig(os.path.join(dataset_dir, 'umap_nsr_developmental.png'), dpi=1200, bbox_inches='tight')
        # plt.show()

        # repeat for tsne with perplexity
        perplexity_experiment = results['perplexity']
        perplexities = []
        spearman = []
        pearson = []
        for perplexity_str, data in perplexity_experiment.items():
            perplexity = float(perplexity_str.replace('perplexity_', ''))
            perplexities.append(perplexity)
            spearman.append(data['spearman_corr'])
            pearson.append(data['pearson_corr'])
        # Sort by perplexity
        sorted_indices = sorted(range(len(perplexities)), key=lambda i: perplexities[i])
        perplexities = [perplexities[i] for i in sorted_indices]
        spearman = [spearman[i] for i in sorted_indices]
        pearson = [pearson[i] for i in sorted_indices]
        print(max(spearman), max(pearson))

        # Create plot
        fig, ax1 = plt.subplots(figsize=(10, 6))
        # Plot Spearman
        color1 = 'tab:blue'
        ax1.set_xlabel('t-SNE perplexity', fontsize=16)
        ax1.set_ylabel('Spearman Correlation', color=color1, fontsize=16)
        ax1.tick_params(axis='y', labelcolor=color1)
        # Twin axis for Pearson
        ax2 = ax1.twinx()
        color2 = 'tab:green'
        ax2.set_ylabel('Pearson Correlation', color=color2, fontsize=16)
        line3 = ax2.plot(perplexities, pearson, marker='s', color=color2, label='t-SNE Pearson', linewidth=3, markersize=8)
        line4 = ax2.axhline(y=embedor_pearson, color=color2, linestyle='--', label='EmbedOR Pearson', linewidth=3, markersize=8)
        ax2.tick_params(axis='y', labelcolor=color2)
        line1 = ax1.plot(perplexities, spearman, marker='o', color=color1, label='t-SNE Spearman', linewidth=3, markersize=8)
        line2 = ax1.axhline(y=embedor_spearman, color=color1, linestyle='--', label='EmbedOR Spearman', linewidth=3, markersize=8)
        # Align y-axis ticks: 0.3 to 0.7 in steps of 0.05
        yticks = np.arange(0.25, 0.75, 0.05)
        xticks = perplexities
        ax1.set_yticks(yticks)
        ax1.set_ylim(0.25, 0.7)
        ax1.set_xticks(xticks)
        ax2.set_ylim(0.25, 0.7)
        ax2.grid(False)
        # Combine legends
        lines = line1 + [line2] + line3 + [line4]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='lower right', framealpha=1.0, facecolor='white', edgecolor='black')
        ax2.legend(lines, labels, loc='lower right', framealpha=1.0, facecolor='white', edgecolor='black')
        plt.tight_layout()
        plt.savefig(os.path.join(dataset_dir, 'tsne_perplexity_developmental.png'), dpi=1200, bbox_inches='tight')
        # plt.show()

        # repeat for tsne with early exaggeration (ee)
        ee_experiment = results['ee']
        ees = []
        spearman = []
        pearson = []
        for ee_str, data in ee_experiment.items():
            ee = float(ee_str.replace('ee_', ''))
            ees.append(ee)
            spearman.append(data['spearman_corr'])
            pearson.append(data['pearson_corr'])
        # Sort by ee
        sorted_indices = sorted(range(len(ees)), key=lambda i: ees[i])
        ees = [ees[i] for i in sorted_indices]
        spearman = [spearman[i] for i in sorted_indices]
        pearson = [pearson[i] for i in sorted_indices]
        print(max(spearman), max(pearson))
        # Create plot
        fig, ax1 = plt.subplots(figsize=(10, 6))
        # Plot Spearman
        color1 = 'tab:blue'
        ax1.set_xlabel('t-SNE early exaggeration', fontsize=16)
        ax1.set_ylabel('Spearman Correlation', color=color1, fontsize=16)
        ax1.tick_params(axis='y', labelcolor=color1)
        # Twin axis for Pearson
        ax2 = ax1.twinx()
        color2 = 'tab:green'
        ax2.set_ylabel('Pearson Correlation', color=color2, fontsize=16)
        line3 = ax2.plot(ees, pearson, marker='s', color=color2, label='t-SNE Pearson', linewidth=3, markersize=8)
        line4 = ax2.axhline(y=embedor_pearson, color=color2, linestyle='--', label='EmbedOR Pearson', linewidth=3, markersize=8)
        ax2.tick_params(axis='y', labelcolor=color2)
        line1 = ax1.plot(ees, spearman, marker='o', color=color1, label='t-SNE Spearman', linewidth=3, markersize=8)
        line2 = ax1.axhline(y=embedor_spearman, color=color1, linestyle='--', label='EmbedOR Spearman', linewidth=3, markersize=8)
        # Align y-axis ticks: 0.3 to 0.7 in steps of 0.05
        yticks = np.arange(0.25, 0.75, 0.05)
        xticks = ees
        ax1.set_yticks(yticks)
        ax1.set_ylim(0.25, 0.7)
        ax1.set_xticks(xticks)
        ax2.set_ylim(0.25, 0.7)
        ax2.grid(False)
        # Combine legends
        lines = line1 + [line2] + line3 + [line4]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='lower right', framealpha=1.0, facecolor='white', edgecolor='black')
        ax2.legend(lines, labels, loc='lower right', framealpha=1.0, facecolor='white', edgecolor='black')
        plt.tight_layout()
        plt.savefig(os.path.join(dataset_dir, 'tsne_ee_developmental.png'), dpi=1200, bbox_inches='tight')
        # plt.show()

    umap_min_dist_experiment = results['min_dist']

    embedor_z_scores_low_energy = results['embedor']['z_scores_low_energy']
    embedor_z_scores_low_energy_mean = np.mean(embedor_z_scores_low_energy)
    embedor_z_scores_low_energy_std = np.std(embedor_z_scores_low_energy)

    min_dists = []
    z_scores_low_energy_means = []
    z_scores_low_energy_stds = []

    for min_dist_str, data in umap_min_dist_experiment.items():
        min_dist = float(min_dist_str.replace('min_dist_', ''))
        min_dists.append(min_dist)
        z_scores_low_energy = data['z_scores_low_energy']
        z_scores_low_energy_mean = np.mean(z_scores_low_energy)
        z_scores_low_energy_means.append(z_scores_low_energy_mean)
        z_scores_low_energy_std = np.std(z_scores_low_energy)
        z_scores_low_energy_stds.append(z_scores_low_energy_std)
        t_stat, p_val = ttest_rel(embedor_z_scores_low_energy, z_scores_low_energy, alternative='less')
        print("*"*100)
        print(f"min_dist: {min_dist}")
        print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
        f"t = {t_stat:.4f}, p = {p_val:.4g} — "
        f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")

        print("*"*100)
        print()

    # plot with fill_between
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(min_dists, z_scores_low_energy_means, marker='o', color='tab:blue', label=r'UMAP Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(min_dists,
                    np.array(z_scores_low_energy_means) - np.array(z_scores_low_energy_stds),
                    np.array(z_scores_low_energy_means) + np.array(z_scores_low_energy_stds),
                    color='tab:blue', alpha=0.2, label=r'UMAP StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')
    ax.set_xlabel('UMAP min_dist', fontsize=16)
    ax.set_ylabel('Z-Scored distance of Low $\Delta^{\mathcal{E}}$ edges', fontsize=16)
    ax.set_ylim(-1.25, 0.3)
    ax.set_xticks([0.01, 0.1, 0.2, 0.5, 1.0])
    ax.set_xticklabels([0.01, 0.1, 0.2, 0.5, 1.0], fontsize=12)

    # plot embedor z_scores_low_energy with different color, same format
    ax.plot(min_dists, [embedor_z_scores_low_energy_mean] * len(min_dists),
            color='tab:orange', label=r'EmbedOR Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(min_dists,
                    embedor_z_scores_low_energy_mean - embedor_z_scores_low_energy_std,
                    embedor_z_scores_low_energy_mean + embedor_z_scores_low_energy_std,
                    color='tab:orange', alpha=0.2, label=r'EmbedOR StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')

    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(dataset_dir, 'umap_min_dist_z_scores_low_energy.png'), dpi=1200, bbox_inches='tight')
    plt.show()

    umap_nsr_experiment = results['nsr']

    embedor_z_scores_low_energy = results['embedor']['z_scores_low_energy']
    embedor_z_scores_low_energy_mean = np.mean(embedor_z_scores_low_energy)
    embedor_z_scores_low_energy_std = np.std(embedor_z_scores_low_energy)

    nsrs = []
    z_scores_low_energy_means = []
    z_scores_low_energy_stds = []

    for nsr_str, data in umap_nsr_experiment.items():
        nsr = float(nsr_str.replace('nsr_', ''))
        nsrs.append(nsr)
        z_scores_low_energy = data['z_scores_low_energy']
        z_scores_low_energy_mean = np.mean(z_scores_low_energy)
        z_scores_low_energy_means.append(z_scores_low_energy_mean)
        z_scores_low_energy_std = np.std(z_scores_low_energy)
        z_scores_low_energy_stds.append(z_scores_low_energy_std)
        t_stat, p_val = ttest_rel(embedor_z_scores_low_energy, z_scores_low_energy, alternative='less')
        print("*"*100)
        print(f"nsr: {nsr}")
        print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
        f"t = {t_stat:.4f}, p = {p_val:.4g} — "
        f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")

        print("*"*100)
        print()

    # plot with fill_between
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(nsrs, z_scores_low_energy_means, marker='o', color='tab:blue', label=r'UMAP Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(nsrs,
                    np.array(z_scores_low_energy_means) - np.array(z_scores_low_energy_stds),
                    np.array(z_scores_low_energy_means) + np.array(z_scores_low_energy_stds),
                    color='tab:blue', alpha=0.2, label=r'UMAP StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')
    ax.set_xlabel('UMAP negative sampling rate', fontsize=16)
    ax.set_ylabel(r'Z-Scored distance of Low $\Delta^{\mathcal{E}}$ edges', fontsize=16)
    ax.set_ylim(-1.25, 0.3)
    ax.set_xticks(list(range(11)))
    # ax.set_xticks([0.01, 0.1, 0.2, 0.5, 1.0])
    # ax.set_xticklabels([0.01, 0.1, 0.2, 0.5, 1.0], fontsize=12)

    # plot embedor z_scores_low_energy with different color, same format
    ax.plot(nsrs, [embedor_z_scores_low_energy_mean] * len(nsrs),
            color='tab:orange', label=r'EmbedOR Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(nsrs,
                    embedor_z_scores_low_energy_mean - embedor_z_scores_low_energy_std,
                    embedor_z_scores_low_energy_mean + embedor_z_scores_low_energy_std,
                    color='tab:orange', alpha=0.2, label=r'EmbedOR StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')

    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(dataset_dir, 'umap_nsr_z_scores_low_energy.png'), dpi=1200, bbox_inches='tight')
    plt.show()

    tsne_perplexity_experiment = results['perplexity']

    embedor_z_scores_low_energy = results['embedor']['z_scores_low_energy']
    embedor_z_scores_low_energy_mean = np.mean(embedor_z_scores_low_energy)
    embedor_z_scores_low_energy_std = np.std(embedor_z_scores_low_energy)

    perplexities = []
    z_scores_low_energy_means = []
    z_scores_low_energy_stds = []

    for perplexity_str, data in tsne_perplexity_experiment.items():
        perplexity = float(perplexity_str.replace('perplexity_', ''))
        perplexities.append(perplexity)
        z_scores_low_energy = data['z_scores_low_energy']
        z_scores_low_energy_mean = np.mean(z_scores_low_energy)
        z_scores_low_energy_means.append(z_scores_low_energy_mean)
        z_scores_low_energy_std = np.std(z_scores_low_energy)
        z_scores_low_energy_stds.append(z_scores_low_energy_std)
        t_stat, p_val = ttest_rel(embedor_z_scores_low_energy, z_scores_low_energy, alternative='less')
        print("*"*100)
        print(f"perplexity: {perplexity}")
        print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
        f"t = {t_stat:.4f}, p = {p_val:.4g} — "
        f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")

        print("*"*100)
        print()

    # plot with fill_between
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(perplexities, z_scores_low_energy_means, marker='o', color='tab:blue', label=r'tSNE Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(perplexities,
                    np.array(z_scores_low_energy_means) - np.array(z_scores_low_energy_stds),
                    np.array(z_scores_low_energy_means) + np.array(z_scores_low_energy_stds),
                    color='tab:blue', alpha=0.2, label=r'tSNE StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')
    ax.set_xlabel('tSNE perplexity', fontsize=16)
    ax.set_xticks(perplexities)
    ax.set_ylabel(r'Z-Scored distance of Low $\Delta^{\mathcal{E}}$ edges', fontsize=16)
    ax.set_ylim(-1.25, 0.3)

    # plot embedor z_scores_low_energy with different color, same format
    ax.plot(perplexities, [embedor_z_scores_low_energy_mean] * len(perplexities),
            color='tab:orange', label=r'EmbedOR Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(perplexities,
                    embedor_z_scores_low_energy_mean - embedor_z_scores_low_energy_std,
                    embedor_z_scores_low_energy_mean + embedor_z_scores_low_energy_std,
                    color='tab:orange', alpha=0.2, label=r'EmbedOR StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')

    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(dataset_dir, 'tsne_perplexity_z_scores_low_energy.png'), dpi=1200, bbox_inches='tight')
    plt.show()

    tsne_ee_experiment = results['ee']

    embedor_z_scores_low_energy = results['embedor']['z_scores_low_energy']
    embedor_z_scores_low_energy_mean = np.mean(embedor_z_scores_low_energy)
    embedor_z_scores_low_energy_std = np.std(embedor_z_scores_low_energy)

    ees = []
    z_scores_low_energy_means = []
    z_scores_low_energy_stds = []

    for ee_str, data in tsne_ee_experiment.items():
        ee = float(ee_str.replace('ee_', ''))
        ees.append(ee)
        z_scores_low_energy = data['z_scores_low_energy']
        z_scores_low_energy_mean = np.mean(z_scores_low_energy)
        z_scores_low_energy_means.append(z_scores_low_energy_mean)
        z_scores_low_energy_std = np.std(z_scores_low_energy)
        z_scores_low_energy_stds.append(z_scores_low_energy_std)
        t_stat, p_val = ttest_rel(embedor_z_scores_low_energy, z_scores_low_energy, alternative='less')
        print("*"*100)
        print(f"early exaggeration: {ee}")
        print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
        f"t = {t_stat:.4f}, p = {p_val:.4g} — "
        f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")

        print("*"*100)
        print()

    # plot with fill_between
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(ees, z_scores_low_energy_means, marker='o', color='tab:blue', label=r'tSNE Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(ees,
                    np.array(z_scores_low_energy_means) - np.array(z_scores_low_energy_stds),
                    np.array(z_scores_low_energy_means) + np.array(z_scores_low_energy_stds),
                    color='tab:blue', alpha=0.2, label=r'tSNE StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')
    ax.set_xlabel('tSNE early exaggeration', fontsize=16)
    ax.set_xticks(ees)
    ax.set_ylabel(r'Z-Scored distance of Low $\Delta^{\mathcal{E}}$ edges', fontsize=16)
    ax.set_ylim(-1.25, 0.3)

    # plot embedor z_scores_low_energy with different color, same format
    ax.plot(ees, [embedor_z_scores_low_energy_mean] * len(ees),
            color='tab:orange', label=r'EmbedOR Mean Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges', linewidth=3, markersize=8)
    ax.fill_between(ees,
                    embedor_z_scores_low_energy_mean - embedor_z_scores_low_energy_std,
                    embedor_z_scores_low_energy_mean + embedor_z_scores_low_energy_std,
                    color='tab:orange', alpha=0.2, label=r'EmbedOR StdDev Z-Score distance of Low $\Delta^{\mathcal{E}}$ edges')

    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(dataset_dir, 'tsne_ee_z_scores_low_energy.png'), dpi=1200, bbox_inches='tight')
    plt.show()


